# HGDR demo walkthrough

Runs the Phase-2 pipeline on the **public MIMIC-III demo** (100 patients) from Python, step by step.
Outputs are stripped; numbers on the demo are not meaningful.

Prerequisite: run `powershell -File scripts/demo_mode.ps1` once (downloads the demo into `data/raw/mimic-iii-demo`), or point `DEMO_DIR` at your own copy.

In [ ]:
import sys, json, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
os.chdir(ROOT)
DEMO_DIR = Path('data/raw/mimic-iii-demo')
OUT = Path('data/processed/demo')
print(ROOT, DEMO_DIR.exists())

## 1. Preprocess MIMIC tables into admission records

In [ ]:
from hgdr.data.mimic import preprocess_mimic
data = preprocess_mimic(DEMO_DIR, min_drug_admissions=3, min_diag_count=2, min_proc_count=2)
data.save(OUT)
print(json.dumps(data.stats, indent=1)[:800])
print(data.drug_voc.idx2word[:20])

## 2. Build the heterogeneous graph (DDI edges from TWOSIDES, molecules from SMILES)

In [ ]:
import numpy as np, pandas as pd
from hgdr.data.drug_mapping import load_mapping
from hgdr.data.ddi import build_ddi_adjacency
from hgdr.data.graph import build_hetero_graph
from hgdr.data.splits import split_records
mapping = load_mapping('data/mappings/drug_name_to_pubchem.json')
pairs = pd.read_csv('data/mappings/twosides_pairs_top40.csv')
cid = {w: int(mapping[w]['cid']) for w in data.drug_voc.idx2word if w in mapping and mapping[w].get('cid')}
ddi = build_ddi_adjacency(data.drug_voc.idx2word, cid, pairs)
train, val, test = split_records(data.records, [0.8, 0.1, 0.1], 0)
graph = build_hetero_graph(train, len(data.diag_voc), len(data.proc_voc), data.drug_voc.idx2word, mapping, ddi,
                           min_cooccur=2, top_k_per_node=20, drug_cooccur_min=2)
graph.save(OUT / 'graph.pt')
print(graph.summary())

## 3. Train the full model for a few epochs

In [ ]:
from hgdr.config import load_config
from hgdr.train import run_experiment
cfg = load_config('configs/demo.yaml', ['train.epochs=8'])
out = run_experiment(cfg, run_name='notebook_demo', results_dir='results_demo')
out['test']

## 4. Inspect recommendations for one test admission

In [ ]:
import torch
from hgdr.train import load_data, make_loaders, predict
from hgdr.models import build_model
data2, graph2, tr, va, te = load_data(cfg)
loaders = make_loaders(cfg, graph2.num_nodes['drug'], tr, va, te)
model = build_model(cfg, graph2)
model.load_state_dict(torch.load('results_demo/runs/notebook_demo/best.pt', map_location='cpu', weights_only=True))
y, p = predict(model, loaders[2], torch.device('cpu'))
i = 0
truth = [data2.drug_voc.idx2word[j] for j in np.nonzero(y[i])[0]]
rec = [data2.drug_voc.idx2word[j] for j in np.argsort(-p[i])[:10]]
print('prescribed :', truth)
print('top-10 rec :', rec)

## 5. Ablation on the demo (2 variants, 1 seed) and figures

For the real study use `python scripts/run_ablation.py` on the full MIMIC-III processed data.

In [ ]:
!python scripts/run_ablation.py --processed_dir data/processed/demo --results_dir results_demo --variants full no_ddi --seeds 0 --set train.epochs=5 model.hidden_dim=32 train.batch_size=32 "train.select_metric=jaccard+prauc"
!python scripts/make_figures.py --results_dir results_demo